# Automation market: pricing, architecture, and protocol comparison

**Claim cluster:** `C001`, `C002`, `C003`, `C004`, `C005`, `C006`, `C007`, `C017`, `C018`, `C019`, `C020`, `C021`, `C022`, `C023`, `C024`, `C084`, `C085`, `C086`  
**Source of truth:** `data/AutomationResearch.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "AutomationResearch.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/AutomationResearch.duckdb (read-only).
CLAIM_IDS = ('C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C017', 'C018', 'C019', 'C020', 'C021', 'C022', 'C023', 'C024', 'C084', 'C085', 'C086')
assert viz.DEFAULT_DB_PATH.endswith("AutomationResearch.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

This cluster brings together the market thesis (C001), the pricing snapshot (C002-C007), the MCP/A2A interoperability claims (C017-C024), and the architecture principles (C084-C086). The claims span different confidence levels — pricing claims are mostly `documented fact` with high confidence, while interoperability and architecture claims include both `documented fact` and `reported signal`.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Pricing, architecture, and protocol claims across the market

C001 establishes that automation is a stack, not a market. C002-C007 give documented pricing for six services. C017-C024 compare MCP (agent-to-tool) with A2A (agent-to-agent) and describe the adapter layer needed to combine them. C084-C086 state architecture principles about side-effect boundaries and total cost.

In [ ]:
# Cluster claim count and confidence mix
_cluster_counts = claims.groupby("claim_type")["local_id"].count().reset_index()
_cluster_counts.columns = ["type", "count"]
show(_cluster_counts)

# Claim texts for the key pricing / architecture / protocol claims
for _cid in ["C001", "C002", "C003", "C004", "C005", "C006", "C007",
             "C017", "C018", "C019", "C020"]:
   display(HTML(f"<h4>{_cid}</h4>"))
display(provenance(_cid))
import plotly.graph_objects as go
_prices = pd.DataFrame([
    {"service": "Zapier", "price": 19.99},
    {"service": "Power Automate", "price": 15},
    {"service": "Temporal Cloud", "price": 50},
    {"service": "UiPath Basic", "price": 25},
    {"service": "n8n Pro", "price": 60},
])
fig = go.Figure(go.Bar(x=_prices["service"], y=_prices["price"], marker_color="#1f77b4", text=_prices["price"], textposition="outside"))
fig.update_layout(title="Monthly pricing USD per service (evidence reference)", height=300)
fig.show()

## Supporting context

### Supporting data from the corpus

The metric table records documented prices for Zapier (M2, M18, M19), Power Automate (M3, M4, M5-M8), Temporal (M9-M13), UiPath (M14), and n8n (M15, M16, M64, M65). M1 is the Zapier free-tier count (100 tasks/month). M20-M21 give estimated startup costs for custom agent stacks.

In [ ]:
display(HTML("<h4>Pricing metrics referenced by the cluster claims</h4>"))
_pricing = viz.metrics_by_id(["M1","M2","M3","M4","M9","M14","M15","M20","M25"])
show(_pricing[["local_id","metric_name","value","unit","confidence"]].rename(
    columns={"local_id":"metric","metric_name":"name","confidence":"conf"}))

## Uncertainty and gaps

### What the cluster does not claim

- **Pricing is time-bound.** C087 states prices change frequently; M2, M3, and M4 are documented facts but only at the snapshot date (2026-09-29).
- **C020 (MCP/A2A combined workflow) is a `reported signal`** (medium confidence); the thin-adapter claim is based on a prototype finding, not a production deployment.
- **No per-service feature matrix exists** for all 10 services; only selective claims are recorded.

In [ ]:
# Claims that describe architecture patterns rather than specific services
_ag = ["C084","C085","C086","C090","C095","C097"]
display(HTML("<h4>Architecture / recommendation claims (not tied to a price point)</h4>"))
show(claims.loc[claims["local_id"].isin(_ag),
                ["local_id","claim_type","confidence","claim_text"]].rename(
    columns={"local_id":"claim","claim_type":"type","confidence":"conf"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Claim ids from the extraction (C001-C007, C017-C024, C084-C086) are verified against `viz.fetch_claims`; pricing numbers from `viz.metrics_by_id` with explicit ids. The claim-to-metric links are authored (C002 -> M2, C003 -> M3, etc.) and shown above rather than stored in the database.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()